# Motion Pickup Debug Demo

This notebook is for a hardcoded floor-demo flow: verify DepthNet, simulate target scanning, drive to the can with fixed timing, run the tuned grab sequence, confirm pickup with a depth threshold, drive to the drop point, and release.

Grab parameters are loaded from `arm_grab_tuning_params.json`. Motion, drop, depth threshold, and A/B/C floor-point parameters are loaded from `motion_pickup_demo_params.json`.


In [ ]:
import json
import time
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

from depth_camera import DepthCamera, mean_of_stats

GRAB_PARAM_PATH = Path('arm_grab_tuning_params.json')
MOTION_PARAM_PATH = Path('motion_pickup_demo_params.json')

DEFAULT_GRAB_PARAMS = {
    'safe_s1': 0, 'safe_s2': 0, 'safe_s3': 0, 'safe_s4': 0, 'safe_s5': 0,
    'ready_s1': 0, 'ready_s2': 18, 'ready_s3': -12, 'ready_s4': 0, 'ready_s5': 0,
    'pre_s2': 18, 'pre_s3': -12,
    'reach_s2': 26, 'reach_s3': -22,
    'lift_s2': 5, 'lift_s3': 18,
    'gripper_open': 0,
    'gripper_close_1': -45,
    'gripper_close_2': -55,
    'arm_speed': 80,
    'gripper_speed': 120,
    'settle_seconds': 0.35,
}

DEFAULT_MOTION_PARAMS = {
    'dry_run_base': True,
    'dry_run_arm': True,
    'skip_depth_check': False,
    'depth_width': 320,
    'depth_height': 240,
    'depth_network': 'fcn-mobilenet',
    'depth_region_x1': 0.40,
    'depth_region_y1': 0.40,
    'depth_region_x2': 0.60,
    'depth_region_y2': 0.60,
    'pickup_success_depth_threshold': 1.60,
    'monitor_interval_seconds': 0.75,
    'start_x': 0.0,
    'start_y': 0.0,
    'can_x': 1.0,
    'can_y': 0.0,
    'drop_x': 1.0,
    'drop_y': 1.0,
    'scan_turn_direction': 'left',
    'scan_turn_speed': 0.12,
    'scan_turn_seconds': 1.0,
    'scan_stop_angle_deg': 90,
    'approach_direction': 'forward',
    'approach_speed': 0.15,
    'approach_seconds': 1.2,
    'approach_distance_m': 0.5,
    'drop_turn_direction': 'right',
    'drop_turn_speed': 0.12,
    'drop_turn_seconds': 1.0,
    'drop_stop_angle_deg': 90,
    'drop_drive_direction': 'forward',
    'drop_drive_speed': 0.15,
    'drop_drive_seconds': 1.2,
    'drop_drive_distance_m': 0.5,
    'pose_after_grab_s1': 0,
    'pose_after_grab_s2': 8,
    'pose_after_grab_s3': 10,
    'pose_after_grab_s4': -55,
    'pose_after_grab_s5': 0,
    'drop_pose_s1': 0,
    'drop_pose_s2': 18,
    'drop_pose_s3': -10,
    'drop_pose_s4': -55,
    'drop_pose_s5': 0,
    'release_grip': 0,
    'arm_speed': 80,
    'gripper_speed': 120,
    'settle_seconds': 0.35,
}


def load_json_params(path, defaults):
    params = dict(defaults)
    if path.exists():
        params.update(json.loads(path.read_text()))
    return params


grab_params = load_json_params(GRAB_PARAM_PATH, DEFAULT_GRAB_PARAMS)
motion_params = load_json_params(MOTION_PARAM_PATH, DEFAULT_MOTION_PARAMS)

print('[params] grab:', GRAB_PARAM_PATH if GRAB_PARAM_PATH.exists() else 'defaults')
print('[params] motion:', MOTION_PARAM_PATH if MOTION_PARAM_PATH.exists() else 'defaults')


In [ ]:
def int_slider(name, params, min_value=-180, max_value=180, step=1, width='520px'):
    return widgets.IntSlider(
        value=int(params[name]), min=min_value, max=max_value, step=step,
        description=name, continuous_update=False,
        style={'description_width': '170px'}, layout=widgets.Layout(width=width)
    )


def float_slider(name, params, min_value=0.0, max_value=2.0, step=0.01, width='520px'):
    return widgets.FloatSlider(
        value=float(params[name]), min=min_value, max=max_value, step=step,
        description=name, continuous_update=False, readout_format='.2f',
        style={'description_width': '170px'}, layout=widgets.Layout(width=width)
    )


def direction_dropdown(name, params):
    return widgets.Dropdown(
        value=params[name], options=['forward', 'backward', 'left', 'right'],
        description=name, style={'description_width': '170px'}, layout=widgets.Layout(width='360px')
    )


def turn_dropdown(name, params):
    return widgets.Dropdown(
        value=params[name], options=['left', 'right'],
        description=name, style={'description_width': '170px'}, layout=widgets.Layout(width='360px')
    )

motion_widgets = {
    'dry_run_base': widgets.Checkbox(value=bool(motion_params['dry_run_base']), description='dry_run_base'),
    'dry_run_arm': widgets.Checkbox(value=bool(motion_params['dry_run_arm']), description='dry_run_arm'),
    'skip_depth_check': widgets.Checkbox(value=bool(motion_params.get('skip_depth_check', False)), description='skip_depth_check'),
    'depth_region_x1': float_slider('depth_region_x1', motion_params, 0.0, 1.0, 0.01),
    'depth_region_y1': float_slider('depth_region_y1', motion_params, 0.0, 1.0, 0.01),
    'depth_region_x2': float_slider('depth_region_x2', motion_params, 0.0, 1.0, 0.01),
    'depth_region_y2': float_slider('depth_region_y2', motion_params, 0.0, 1.0, 0.01),
    'pickup_success_depth_threshold': float_slider('pickup_success_depth_threshold', motion_params, 0.1, 5.0, 0.05),
    'monitor_interval_seconds': float_slider('monitor_interval_seconds', motion_params, 0.1, 3.0, 0.05),
    'start_x': float_slider('start_x', motion_params, -5.0, 5.0, 0.05),
    'start_y': float_slider('start_y', motion_params, -5.0, 5.0, 0.05),
    'can_x': float_slider('can_x', motion_params, -5.0, 5.0, 0.05),
    'can_y': float_slider('can_y', motion_params, -5.0, 5.0, 0.05),
    'drop_x': float_slider('drop_x', motion_params, -5.0, 5.0, 0.05),
    'drop_y': float_slider('drop_y', motion_params, -5.0, 5.0, 0.05),
    'scan_turn_direction': turn_dropdown('scan_turn_direction', motion_params),
    'scan_turn_speed': float_slider('scan_turn_speed', motion_params, 0.0, 0.5, 0.01),
    'scan_turn_seconds': float_slider('scan_turn_seconds', motion_params, 0.0, 5.0, 0.05),
    'scan_stop_angle_deg': int_slider('scan_stop_angle_deg', motion_params, 0, 360, 5),
    'approach_direction': direction_dropdown('approach_direction', motion_params),
    'approach_speed': float_slider('approach_speed', motion_params, 0.0, 0.5, 0.01),
    'approach_seconds': float_slider('approach_seconds', motion_params, 0.0, 8.0, 0.05),
    'approach_distance_m': float_slider('approach_distance_m', motion_params, 0.0, 5.0, 0.05),
    'drop_turn_direction': turn_dropdown('drop_turn_direction', motion_params),
    'drop_turn_speed': float_slider('drop_turn_speed', motion_params, 0.0, 0.5, 0.01),
    'drop_turn_seconds': float_slider('drop_turn_seconds', motion_params, 0.0, 5.0, 0.05),
    'drop_stop_angle_deg': int_slider('drop_stop_angle_deg', motion_params, 0, 360, 5),
    'drop_drive_direction': direction_dropdown('drop_drive_direction', motion_params),
    'drop_drive_speed': float_slider('drop_drive_speed', motion_params, 0.0, 0.5, 0.01),
    'drop_drive_seconds': float_slider('drop_drive_seconds', motion_params, 0.0, 8.0, 0.05),
    'drop_drive_distance_m': float_slider('drop_drive_distance_m', motion_params, 0.0, 5.0, 0.05),
    'pose_after_grab_s1': int_slider('pose_after_grab_s1', motion_params),
    'pose_after_grab_s2': int_slider('pose_after_grab_s2', motion_params),
    'pose_after_grab_s3': int_slider('pose_after_grab_s3', motion_params),
    'pose_after_grab_s4': int_slider('pose_after_grab_s4', motion_params),
    'pose_after_grab_s5': int_slider('pose_after_grab_s5', motion_params),
    'drop_pose_s1': int_slider('drop_pose_s1', motion_params),
    'drop_pose_s2': int_slider('drop_pose_s2', motion_params),
    'drop_pose_s3': int_slider('drop_pose_s3', motion_params),
    'drop_pose_s4': int_slider('drop_pose_s4', motion_params),
    'drop_pose_s5': int_slider('drop_pose_s5', motion_params),
    'release_grip': int_slider('release_grip', motion_params),
    'arm_speed': int_slider('arm_speed', motion_params, 20, 300, 5),
    'gripper_speed': int_slider('gripper_speed', motion_params, 20, 300, 5),
    'settle_seconds': float_slider('settle_seconds', motion_params, 0.05, 1.5, 0.05),
}


def current_motion_params():
    return {name: widget.value for name, widget in motion_widgets.items()}


def current_depth_region():
    p = current_motion_params()
    return (p['depth_region_x1'], p['depth_region_y1'], p['depth_region_x2'], p['depth_region_y2'])


def save_motion_params(_=None):
    data = current_motion_params()
    data['skip_depth_check'] = bool(data.get('skip_depth_check', False))
    data['depth_width'] = int(motion_params.get('depth_width', 320))
    data['depth_height'] = int(motion_params.get('depth_height', 240))
    data['depth_network'] = str(motion_params.get('depth_network', 'fcn-mobilenet'))
    MOTION_PARAM_PATH.write_text(json.dumps(data, indent=2))
    print('[params] saved motion params to', MOTION_PARAM_PATH)


In [ ]:
depth = None
robot = None
ttl_servo = None


def start_depth_and_verify(frame_count=2):
    global depth
    if depth is None:
        print('[depth] starting network and camera')
        depth = DepthCamera(
            width=int(motion_params.get('depth_width', 320)),
            height=int(motion_params.get('depth_height', 240)),
            network=str(motion_params.get('depth_network', 'fcn-mobilenet')),
        )
        depth.start(warmup_frames=1)
    print('[depth] verifying {} frames'.format(frame_count))
    stats = []
    for i in range(frame_count):
        item = depth.observe(region=current_depth_region())
        stats.append(item)
        print('[depth] verify frame={} mean={:.3f} min={:.3f} max={:.3f}'.format(
            i + 1, item['mean'], item['min'], item['max']
        ))
        time.sleep(0.2)
    return stats


def stop_depth():
    global depth
    if depth is not None:
        depth.stop()
        depth = None
        print('[depth] stopped')


In [ ]:
def ensure_robot():
    global robot
    if motion_widgets['dry_run_base'].value:
        return None
    if robot is None:
        from jetbot import Robot
        robot = Robot()
        print('[base] Robot connected')
    return robot


def ensure_servos():
    global ttl_servo
    if motion_widgets['dry_run_arm'].value:
        return None
    if ttl_servo is None:
        from SCSCtrl import TTLServo
        ttl_servo = TTLServo
        print('[arm] TTLServo connected')
    return ttl_servo


def base_stop():
    if robot is not None:
        robot.stop()
    print('[base] stop')


def base_drive(direction, speed, seconds, label):
    bot = ensure_robot()
    print('[base] {} direction={} speed={} seconds={}'.format(label, direction, speed, seconds))
    if bot is not None:
        try:
            if direction == 'forward':
                bot.forward(float(speed))
            elif direction == 'backward':
                bot.backward(float(speed))
            elif direction == 'left':
                bot.left(float(speed))
            elif direction == 'right':
                bot.right(float(speed))
            else:
                raise ValueError('unknown base direction: {}'.format(direction))
            time.sleep(float(seconds))
        finally:
            bot.stop()
    else:
        time.sleep(float(seconds))
    print('[base] {} done'.format(label))


def move_servo(servo_id, angle, speed, label=''):
    servos = ensure_servos()
    print('[arm] servo={} angle={} speed={} {}'.format(servo_id, angle, speed, label))
    if servos is not None:
        servos.servoAngleCtrl(int(servo_id), int(angle), 1, int(speed))
    time.sleep(float(motion_widgets['settle_seconds'].value))


def apply_pose(name, pose, speed):
    print('[arm] pose:', name)
    for servo_id, angle in pose:
        move_servo(servo_id, angle, speed, name)


def grab_value(name):
    return grab_params[name]


def ready_state():
    pose = [(1, grab_value('ready_s1')), (2, grab_value('ready_s2')), (3, grab_value('ready_s3')), (4, grab_value('ready_s4')), (5, grab_value('ready_s5'))]
    apply_pose('ready_state', pose, grab_value('arm_speed'))


def safe_home():
    pose = [(1, grab_value('safe_s1')), (2, grab_value('safe_s2')), (3, grab_value('safe_s3')), (4, grab_value('safe_s4')), (5, grab_value('safe_s5'))]
    apply_pose('safe_home', pose, grab_value('arm_speed'))


def open_gripper():
    move_servo(4, grab_value('gripper_open'), grab_value('gripper_speed'), 'open_gripper')


def close_gripper():
    move_servo(4, grab_value('gripper_close_1'), grab_value('gripper_speed'), 'close_1')
    time.sleep(0.5)
    move_servo(4, grab_value('gripper_close_2'), grab_value('gripper_speed'), 'close_2')


def grab_sequence_without_final_home():
    print('[flow] grab sequence using', GRAB_PARAM_PATH)
    ready_state()
    open_gripper()
    apply_pose('pre_grasp', [(2, grab_value('pre_s2')), (3, grab_value('pre_s3'))], grab_value('arm_speed'))
    apply_pose('reach', [(2, grab_value('reach_s2')), (3, grab_value('reach_s3'))], grab_value('arm_speed'))
    close_gripper()
    apply_pose('lift', [(2, grab_value('lift_s2')), (3, grab_value('lift_s3'))], grab_value('arm_speed'))


def after_grab_pose():
    p = current_motion_params()
    pose = [(1, p['pose_after_grab_s1']), (2, p['pose_after_grab_s2']), (3, p['pose_after_grab_s3']), (4, p['pose_after_grab_s4']), (5, p['pose_after_grab_s5'])]
    apply_pose('after_grab_pose', pose, p['arm_speed'])


def drop_release_sequence():
    p = current_motion_params()
    pose = [(1, p['drop_pose_s1']), (2, p['drop_pose_s2']), (3, p['drop_pose_s3']), (4, p['drop_pose_s4']), (5, p['drop_pose_s5'])]
    apply_pose('drop_pose', pose, p['arm_speed'])
    move_servo(4, p['release_grip'], p['gripper_speed'], 'release_grip')


In [ ]:
def observe_pickup_depth(label):
    if current_motion_params().get('skip_depth_check'):
        print('[depth] skip_depth_check=True; assume can was found and pickup succeeded')
        return {'mean': None, 'min': None, 'max': None}, True
    if depth is None:
        raise RuntimeError('run Start Depth + Verify first')
    stats = depth.observe(region=current_depth_region())
    threshold = float(motion_widgets['pickup_success_depth_threshold'].value)
    success = stats['mean'] < threshold
    print('[depth] {} mean={:.3f} min={:.3f} max={:.3f} threshold={:.3f} grabbed={}'.format(
        label, stats['mean'], stats['min'], stats['max'], threshold, success
    ))
    return stats, success


def print_points():
    p = current_motion_params()
    print('[points] A start=({:.2f}, {:.2f}) B can=({:.2f}, {:.2f}) C drop=({:.2f}, {:.2f})'.format(
        p['start_x'], p['start_y'], p['can_x'], p['can_y'], p['drop_x'], p['drop_y']
    ))
    print('[move] scan stop angle={} deg, approach distance={} m'.format(p['scan_stop_angle_deg'], p['approach_distance_m']))
    print('[move] drop turn stop angle={} deg, drop drive distance={} m'.format(p['drop_stop_angle_deg'], p['drop_drive_distance_m']))


def run_motion_pickup_demo(_=None):
    p = current_motion_params()
    print('[flow] motion pickup demo start')
    print('[flow] dry_run_base={} dry_run_arm={} skip_depth_check={}'.format(p['dry_run_base'], p['dry_run_arm'], p.get('skip_depth_check')))
    print_points()
    try:
        if p.get('skip_depth_check'):
            print('[depth] skip_depth_check=True; skip DepthNet startup and assume target is available')
        else:
            start_depth_and_verify(frame_count=2)
        safe_home()
        base_stop()

        print('[flow] rotate in place to simulate target scan')
        base_drive(p['scan_turn_direction'], p['scan_turn_speed'], p['scan_turn_seconds'], 'scan_turn_to_angle_{}'.format(p['scan_stop_angle_deg']))

        print('[flow] hardcoded move from A to B')
        base_drive(p['approach_direction'], p['approach_speed'], p['approach_seconds'], 'approach_can_distance_{}m'.format(p['approach_distance_m']))

        print('[flow] ready state near can')
        ready_state()

        print('[flow] grab can')
        grab_sequence_without_final_home()

        stats, success = observe_pickup_depth('after_grab_lift')
        if not success:
            print('[flow] pickup failed by depth threshold; return to safe_home')
            safe_home()
            return False

        print('[flow] pickup success; move to carrying posture')
        after_grab_pose()

        print('[flow] rotate in place to simulate drop-point scan')
        base_drive(p['drop_turn_direction'], p['drop_turn_speed'], p['drop_turn_seconds'], 'drop_turn_to_angle_{}'.format(p['drop_stop_angle_deg']))

        print('[flow] hardcoded move from B to C')
        base_drive(p['drop_drive_direction'], p['drop_drive_speed'], p['drop_drive_seconds'], 'drive_to_drop_distance_{}m'.format(p['drop_drive_distance_m']))

        print('[flow] release at drop point')
        drop_release_sequence()
        print('[flow] demo done')
        return True
    finally:
        print('[flow] final cleanup: stop base and safe_home')
        base_stop()
        safe_home()


def emergency_stop(_=None):
    print('[stop] emergency stop')
    base_stop()
    safe_home()


In [ ]:
save_button = widgets.Button(description='Save Motion Params', button_style='info')
depth_button = widgets.Button(description='Start Depth + Verify', button_style='primary')
point_button = widgets.Button(description='Print Points', button_style='')
run_button = widgets.Button(description='Run Full Demo', button_style='success')
stop_button = widgets.Button(description='Emergency Stop', button_style='danger')
home_button = widgets.Button(description='Safe Home', button_style='warning')
ready_button = widgets.Button(description='Ready State', button_style='')
clear_log_button = widgets.Button(description='Clear Log', button_style='')

log_output = widgets.Output(layout={
    'border': '1px solid #bbb',
    'height': '260px',
    'overflow_y': 'auto',
    'width': '100%',
})


def run_with_log(func):
    def wrapped(_=None):
        with log_output:
            try:
                func()
            except Exception as exc:
                print('[error]', type(exc).__name__ + ':', exc)
                print('[hint] If this is a camera error, stop other kernels using the camera, restart this kernel, or run: sudo systemctl restart nvargus-daemon')
    return wrapped


def clear_log(_=None):
    log_output.clear_output()

save_button.on_click(run_with_log(lambda: save_motion_params()))
depth_button.on_click(run_with_log(lambda: start_depth_and_verify(frame_count=2)))
point_button.on_click(run_with_log(print_points))
run_button.on_click(run_with_log(lambda: run_motion_pickup_demo()))
stop_button.on_click(run_with_log(lambda: emergency_stop()))
home_button.on_click(run_with_log(safe_home))
ready_button.on_click(run_with_log(ready_state))
clear_log_button.on_click(clear_log)

button_row = widgets.HBox([save_button, depth_button, point_button, home_button, ready_button, run_button, stop_button, clear_log_button])

ui = widgets.VBox([
    button_row,
    widgets.HTML('<b>Log output</b>'),
    log_output,
    widgets.HTML('<b>Safety</b>'),
    widgets.HBox([motion_widgets['dry_run_base'], motion_widgets['dry_run_arm'], motion_widgets['skip_depth_check']]),
    widgets.HTML('<b>Depth confirmation</b>'),
    motion_widgets['pickup_success_depth_threshold'],
    motion_widgets['monitor_interval_seconds'],
    widgets.HBox([motion_widgets['depth_region_x1'], motion_widgets['depth_region_y1']]),
    widgets.HBox([motion_widgets['depth_region_x2'], motion_widgets['depth_region_y2']]),
    widgets.HTML('<b>Hardcoded points on floor</b>'),
    widgets.HBox([motion_widgets['start_x'], motion_widgets['start_y']]),
    widgets.HBox([motion_widgets['can_x'], motion_widgets['can_y']]),
    widgets.HBox([motion_widgets['drop_x'], motion_widgets['drop_y']]),
    widgets.HTML('<b>Scan turn and approach to can</b>'),
    widgets.HBox([motion_widgets['scan_turn_direction'], motion_widgets['scan_stop_angle_deg']]),
    widgets.HBox([motion_widgets['scan_turn_speed'], motion_widgets['scan_turn_seconds']]),
    widgets.HBox([motion_widgets['approach_direction'], motion_widgets['approach_distance_m']]),
    widgets.HBox([motion_widgets['approach_speed'], motion_widgets['approach_seconds']]),
    widgets.HTML('<b>Carry posture after successful grab</b>'),
    motion_widgets['pose_after_grab_s1'], motion_widgets['pose_after_grab_s2'], motion_widgets['pose_after_grab_s3'], motion_widgets['pose_after_grab_s4'], motion_widgets['pose_after_grab_s5'],
    widgets.HTML('<b>Turn and drive to drop point</b>'),
    widgets.HBox([motion_widgets['drop_turn_direction'], motion_widgets['drop_stop_angle_deg']]),
    widgets.HBox([motion_widgets['drop_turn_speed'], motion_widgets['drop_turn_seconds']]),
    widgets.HBox([motion_widgets['drop_drive_direction'], motion_widgets['drop_drive_distance_m']]),
    widgets.HBox([motion_widgets['drop_drive_speed'], motion_widgets['drop_drive_seconds']]),
    widgets.HTML('<b>Drop posture and release</b>'),
    motion_widgets['drop_pose_s1'], motion_widgets['drop_pose_s2'], motion_widgets['drop_pose_s3'], motion_widgets['drop_pose_s4'], motion_widgets['drop_pose_s5'],
    motion_widgets['release_grip'],
    widgets.HTML('<b>Shared motion servo timing</b>'),
    motion_widgets['arm_speed'], motion_widgets['gripper_speed'], motion_widgets['settle_seconds'],
])

display(ui)


## Usage

1. Run the code cells above until the UI appears.
2. Click `Start Depth + Verify` and confirm that two depth frames are printed.
3. Click `Print Points` to inspect the hardcoded A/B/C floor points.
4. First run should keep `dry_run_base` and `dry_run_arm` enabled, then click `Run Full Demo` and inspect the logs.
5. After the parameters look right, disable the relevant dry-run checkbox and run again. Keep your hand ready for `Emergency Stop` during real movement.
